In [1]:
import numpy as np
import os
import atristoolset as ia
import simtools as st
import subprocess


# Notes> we recommend installing anaconda python. Note that this works only with python3.
# If using with your system python distribution make sure to install all necessary packages e.g. with pip or pip3

N_threads = ia.get_threads() # you can also choose to configure the number of threads manually

In [2]:
rootdir = os.environ.get('ATRIS_ROOTDIR', os.getcwd())  # defaults to the current checkout; override with ATRIS_ROOTDIR if needed
if not rootdir.endswith(os.sep):
    rootdir += os.sep
geant4_install_dir = os.environ.get('GEANT4_INSTALL_DIR')
geant4_sh = os.environ.get('GEANT4_SH')
if geant4_install_dir:
    geant4_dir = os.path.join(geant4_install_dir, 'lib', 'cmake', 'Geant4')
    os.environ.setdefault('Geant4_DIR', geant4_dir)
    os.environ.setdefault('CMAKE_PREFIX_PATH', geant4_dir)
elif geant4_sh:
    geant4_prefix = os.path.dirname(os.path.dirname(geant4_sh))
    geant4_dir = os.path.join(geant4_prefix, 'lib', 'cmake', 'Geant4')
    os.environ.setdefault('Geant4_DIR', geant4_dir)
    os.environ.setdefault('CMAKE_PREFIX_PATH', geant4_dir)
ia.geantinstall(rootdir)  # This will check if a Geant4 installation exists, and if not it will create one
ia.atrisinstall(rootdir)  # This will check if an AtRIS installation exists, and if not it will create one


The file "geant4_install_ok" has been found. Continuing without geant4 installation.
If you wish for the script to perform a clean install, please remove the  "geant4_install_ok".
The file "atris_install_ok" has been found. Continuing without atris installation.
If you wish for the script to perform a clean install, please remove the  "atris_install_ok".
Continuing to source geant4.sh .
Sourcing geant4.sh is complete. Returning to root AtRIS folder.
/git/AtRIS2
Setting work directory to ./atris/simulation successful. Continuing to simulation conf.


In [3]:
# Simulation configuration - please provide here your input for the script to prepare the full simulation
# A text interface was choosen for convenience
### Optional configurations are designated with a tripple # comment
##  Mandatory configurations are designated with a double # comment
#   'Better don't change unless you know what you are doing' comments have a single #
# The following present a best selection of settings according to author

# Provide coordinates, watch for ranges. E.g. NYC long=40.6deg, lat=286deg. This is a common source of mistakes.
longitude     = 10.00          ##  from -90deg to 90deg
latitude      = 10.00          ##  from 0deg to 360deg
year          = 2012           ##  from 1951 until 2021 (data for 2022 sol mod parameter is missing)
month         = 5              ##  FROM 1 to 12
day           = 27             ##  from 1 to 28 or 29 or 30 or 31
hour_utc      = 12.            ##  from 0 to 24.
alt_max       = 100.           ### from 10 to 1000 km
alt_step      = 0.2            ### Step size should be between 0.1 and 5.0 km
particles     = "both"         ### Either "protons", or "alphas", or "both"
numberparts   = 10000          ### number of primary particles to be simulated
numberparts   = 1000           ### number of primary particles to be simulated
numberthreads = N_threads      ### The script will try and use the maximum number of threads
numberthreads = 16             ### The script will try and use the maximum number of threads
alpha_factor  = 0.66           ### I reduce the number of alphas per bin since they are more costly to calculate
phys_list     = "FTFP_BERT_HP" ### physics list - change probably not needed.
alt_min       = 0.             #   DO NOT CHANGE
phantoms      = "all"          #   DO NOT CHANGE

# Note that here we are going to connect to NASA's servers to obtain nrlmsis model data for the atmosphere.
# Take care to give proper credit. https://kauai.ccmc.gsfc.nasa.gov/instantrun/msis
os.chdir(rootdir+'atris/simulation/') # script assumes we are here.
ia.create_simconfig(longitude,latitude,year,month,day,hour_utc,alt_min,alt_max,alt_step,particles,phantoms,numberparts,numberthreads,phys_list,alpha_factor)
phi = st.simprep_manager(year,month,day,hour_utc,latitude,longitude,alt_min,alt_max,alt_step)
st.gen_macros(rootdir,numberparts,numberthreads,alpha_factor)

The file simconfig_ok exists. Using existing simulation configuration. To reconfigure, and
provide a new configuration, please remove the simconfig_ok from the simulation subfolder.
currently working in the following folder:  /git/AtRIS2/atris/simulation
this should be the simulation subfolder within the atris directory. If not, please check.
we will request an atmosphere from nasa using the new nrlmsise model using the following parameters:
{'whichMsis': 1, 'outputOptions': '1', 'profileType': 1, 'year': '2012', 'month': '05', 'day': '27', 'timeFlag': 0, 'hour': '12.0', 'geoFlag': 0, 'latitude': '10.00', 'longitude': '10.00', 'height': '500', 'start': '0', 'stop': '100', 'stepSize': '0.2', 'f107': -1, 'f107month': -1, 'ap': -1}
COMPLETE: writing out the psf lines.
	total mass is not equal to 1!!
			formula: #O478Si284Al84Fe52Ca37Na28Mg15
			totalmas: 0.9780000000. Rescaling.
	total mass is now fine. Problem solved.
converted phi obtained: 628.93 MV for the GCR spectrum (Herbst convers

In [ ]:
# simulation start for protons
st.start_sim(rootdir,'protons',phys_list)
st.monitor_sim('protons')

In [ ]:
# once protons are done, you should continue with alphas. These two cells can be joined to avoid wait time, since the monitor
# function will block the code until the protons simulation is done. 
st.start_sim(rootdir,'alphas',phys_list)
st.monitor_sim('alphas')


In [4]:
# now we do data post processing. This will take up to a minute.
# This function requires cell number 3 to be executed since it needs phi. 
st.unite(rootdir,phi)

This run included 32 phantoms
This simulation had 121 energy bins
we have 32 phantoms in this run
Processing phantoms
Reading data complete.
[ 962. 1081. 1044. 1022.  998. 1002.  999.  979. 1016.  962.  963. 1007.
 1065.  986.  990. 1018.  990.  949. 1042.  973. 1012.  970.  942.  981.
 1072.  961. 1021. 1020. 1035. 1019. 1023. 1040. 1020.  944.  990. 1078.
  995.  933. 1026. 1029. 1044. 1067.  980.  964. 1003.  982. 1014. 1030.
 1018. 1018. 1020. 1064. 1058.  976.  993.  946. 1026. 1026. 1041.  972.
  982.  997. 1018. 1015. 1046.  967. 1044. 1040. 1031.  986. 1018. 1026.
 1052. 1034. 1009. 1009. 1032.  988. 1033.  974.  970.  983. 1023. 1006.
 1021. 1014. 1089. 1017.  983.  990. 1047.  985. 1017. 1018.  992.  974.
  989.  980. 1002.  993. 1018. 1065. 1027.  971. 1039.  978. 1016. 1034.
  986.  977. 1018. 1000.  982. 1006. 1009. 1008.  972. 1001. 1097. 1001.]
Making output files.
Response files have been created.
This run included 32 phantoms
This simulation had 121 energy bins
we have